# NILM Phase 4 — Pruned Window Knowledge (12-D) + Appliance-Level Knowledge Profiles
## (Power / Temporal / Usage) + Compatibility Fusion + Threshold Tuning

**Changes from Phase 3:**
1. Window knowledge vector pruned from 21 -> **12 non-redundant features** (dropped features
   that were mathematically near-duplicates, e.g. `energy_per_window` ~ `mean_power` exactly).
2. Added **appliance-level knowledge profiles**: `K_power`, `K_temporal`, `K_usage` per
   appliance, computed **only from the training split**, then frozen (non-trainable) —
   no val/test leakage.
3. Replaced the single shared 5-way sigmoid head with a **per-appliance compatibility branch**:
   each appliance compares the current window against its own frozen profile before deciding
   ON/OFF, instead of all 5 appliances sharing one bottleneck layer.
4. Added **per-appliance decision threshold tuning** on the validation split (never on test).


## 1. Install & Imports

In [ ]:
!pip install -q tensorflow scikit-learn scipy pandas numpy matplotlib seaborn

In [ ]:
import os, json, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import medfilt
from scipy.stats import skew, kurtosis as kurtosis_fn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (precision_score, recall_score, f1_score,
                              accuracy_score, classification_report)
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

tf.random.set_seed(42)
np.random.seed(42)
print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


## 2. Mount Drive & Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CONFIG = {
    "house_dir": "/content/drive/MyDrive/SEM7PROJ/house_1",
    "checkpoint_dir": "/content/drive/MyDrive/SEM7PROJ/checkpoints_phase4_profile_fusion",
    "sample_period": "6s",
    "sample_seconds": 6,
    "window_size": 100,
    "window_stride": 20,
    "median_kernel": 5,
    "on_thresholds": {"dishwasher": 10, "washing machine": 20, "laptop": 15, "fridge": 20, "office lamp 1": 5},
    "min_on_samples": {"dishwasher": 20, "washing machine": 10, "laptop": 5, "fridge": 5, "office lamp 1": 3},
    "label_rule": {"dishwasher": "majority", "washing machine": "majority", "laptop": "majority",
                   "fridge": "majority", "office lamp 1": "majority"},
    "appliances": ["dishwasher", "washing machine", "laptop", "fridge", "office lamp 1"],
    "batch_size": 64,
    "epochs": 50,
    "min_dwell_windows": 2,
    "duty_loss_weight": 1.0,
    "knowledge_recon_loss_weight": 0.3,
}
os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)
print("Config ready.")

Mounted at /content/drive
Config ready.


## 3. Dataset Loading

In [ ]:
def load_labels(house_dir):
    labels = {}
    with open(os.path.join(house_dir, "labels.dat"), "r") as f:
        for line in f:
            parts = line.strip().split(" ", 1)
            if len(parts) == 2:
                ch_num, name = parts
                labels[int(ch_num)] = name.strip().lower().replace("_", " ")
    return labels

def load_channel(house_dir, channel_num):
    path_dat = os.path.join(house_dir, f"channel_{channel_num}.dat")
    path_csv = os.path.join(house_dir, f"channel_{channel_num}.csv")
    if os.path.exists(path_dat):
        df = pd.read_csv(path_dat, sep=" ", header=None, names=["timestamp", "power"])
    elif os.path.exists(path_csv):
        df = pd.read_csv(path_csv, header=None, names=["timestamp", "power"])
    else:
        raise FileNotFoundError(f"No data file for channel {channel_num}")
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s")
    df = df.set_index("timestamp").sort_index()
    df["power"] = pd.to_numeric(df["power"], errors="coerce")
    return df

labels_map = load_labels(CONFIG["house_dir"])
print(f"Found {len(labels_map)} channels.")

Found 53 channels.


## 4. Appliance Selection & Synthetic Aggregate

In [ ]:
def find_channel_for_appliance(labels_map, appliance_name):
    appliance_name = appliance_name.lower()
    candidates = [ch for ch, name in labels_map.items() if appliance_name in name or name in appliance_name]
    if not candidates:
        words = appliance_name.split()
        candidates = [ch for ch, name in labels_map.items() if all(w in name for w in words)]
    if not candidates:
        raise ValueError(f"No channel found for '{appliance_name}'")
    return candidates[0]

selected_channels, appliance_data = {}, {}
for app in CONFIG["appliances"]:
    ch = find_channel_for_appliance(labels_map, app)
    selected_channels[app] = ch
    df = load_channel(CONFIG["house_dir"], ch)
    appliance_data[app] = df
    print(f"{app:20s} -> channel {ch} | {len(df)} samples")

def resample_series(df, period, origin):
    return df["power"].resample(period, origin=origin).mean()

shared_origin = min(df.index.min() for df in appliance_data.values())
resampled = {app: resample_series(df, CONFIG["sample_period"], shared_origin) for app, df in appliance_data.items()}
combined = pd.DataFrame(resampled).dropna(how="all")
combined = combined.ffill(limit=int(60 / 6))
combined["aggregate"] = combined[CONFIG["appliances"]].sum(axis=1, skipna=True)
print("Aggregate built. Total samples:", len(combined))

dishwasher           -> channel 6 | 19819392 samples
washing machine      -> channel 5 | 19555935 samples
laptop               -> channel 4 | 4539118 samples
fridge               -> channel 12 | 19381298 samples
office lamp 1        -> channel 48 | 18179281 samples
Aggregate built. Total samples: 22736289


## 5. Preprocessing

In [ ]:
data = combined.dropna(subset=CONFIG["appliances"] + ["aggregate"]).copy()

def median_filter_series(series, kernel_size):
    if kernel_size % 2 == 0:
        kernel_size += 1
    return pd.Series(medfilt(series.values, kernel_size=kernel_size), index=series.index)

filtered = data.copy()
for col in CONFIG["appliances"] + ["aggregate"]:
    filtered[col] = median_filter_series(data[col], CONFIG["median_kernel"])
filtered[filtered < 0] = 0
print("Preprocessing complete. Rows:", len(filtered))

Preprocessing complete. Rows: 3930087


## 6. ON/OFF Labels & Day-Cycled Train/Val/Test Split

In [ ]:
def compute_on_mask(power, threshold, min_samples):
    raw_on = power > threshold
    if min_samples <= 1:
        return raw_on
    groups = (raw_on != raw_on.shift()).cumsum()
    run_lengths = raw_on.groupby(groups).transform("size")
    return raw_on & (run_lengths >= min_samples)

for app in CONFIG["appliances"]:
    filtered[f"{app}_on"] = compute_on_mask(filtered[app], CONFIG["on_thresholds"][app], CONFIG["min_on_samples"][app])

filtered["date"] = filtered.index.normalize()
unique_days = filtered["date"].drop_duplicates().sort_values().reset_index(drop=True)
day_split = ["train" if i % 10 < 7 else ("val" if i % 10 == 7 else "test") for i in range(len(unique_days))]
day_assignment = pd.Series(day_split, index=unique_days)
filtered["split"] = filtered["date"].map(day_assignment)

train_df = filtered[filtered["split"] == "train"].drop(columns=["date", "split"])
val_df = filtered[filtered["split"] == "val"].drop(columns=["date", "split"])
test_df = filtered[filtered["split"] == "test"].drop(columns=["date", "split"])
print(f"Rows -> train: {len(train_df)} | val: {len(val_df)} | test: {len(test_df)}")

Rows -> train: 2739242 | val: 417176 | test: 773669


## 7. Pruned Window Knowledge Vector (12-D)
Dropped from the earlier 21-D set: `max_power`, `median_power`, `rms_power`, `range_power`,
`energy_per_window` (all >0.87 correlated with `mean_power`/each other), and `fall_time`
(perfectly anti-correlated with `rise_time` — mathematically redundant given fixed window
length). All remaining features are computed purely from the aggregate window, so they are
leakage-safe on train/val/test alike.

In [ ]:
KNOWLEDGE_FEATURE_NAMES = [
    "mean_power", "std_power", "min_power", "power_jump", "num_transitions",
    "rise_time", "duty_cycle", "iqr_power", "autocorr_peak_strength",
    "skewness", "kurtosis", "dominant_period",
]
N_KNOWLEDGE_FEATURES = len(KNOWLEDGE_FEATURE_NAMES)
assert N_KNOWLEDGE_FEATURES == 12

def compute_window_knowledge(window_agg, sample_seconds=6, edge_threshold_frac=0.1):
    w = window_agg.astype(np.float64)
    n = len(w)
    mean_power = float(np.mean(w))
    min_power = float(np.min(w))
    max_power = float(np.max(w))
    std_power = float(np.std(w))
    range_power = max_power - min_power
    q75, q25 = np.percentile(w, [75, 25])
    iqr_power = float(q75 - q25)

    diffs = np.diff(w) if n > 1 else np.array([0.0])
    power_jump = float(np.max(np.abs(diffs)))
    edge_thresh = max(edge_threshold_frac * range_power, 5.0)
    num_transitions = int(np.sum(np.abs(diffs) > edge_thresh)) if n > 1 else 0

    if n > 4 and std_power > 1e-6:
        w_centered = w - mean_power
        acf = np.correlate(w_centered, w_centered, mode="full")[n - 1:]
        acf = acf / (acf[0] + 1e-8)
        search_start = max(2, n // 20)
        if search_start < len(acf) - 1:
            peak_lag = int(np.argmax(acf[search_start:])) + search_start
            dominant_period = float(peak_lag * sample_seconds)
            autocorr_peak_strength = float(acf[peak_lag])
        else:
            dominant_period, autocorr_peak_strength = 0.0, 0.0
    else:
        dominant_period, autocorr_peak_strength = 0.0, 0.0

    peak_idx = int(np.argmax(w))
    rise_time = float(peak_idx * sample_seconds)
    duty_thresh = min_power + 0.5 * range_power
    duty_cycle = float(np.mean(w > duty_thresh)) if range_power > 0 else 0.0

    if std_power > 1e-6:
        skewness, kurt = float(skew(w)), float(kurtosis_fn(w))
    else:
        skewness, kurt = 0.0, 0.0

    return [mean_power, std_power, min_power, power_jump, num_transitions,
            rise_time, duty_cycle, iqr_power, autocorr_peak_strength, skewness, kurt, dominant_period]

print("Pruned knowledge feature names (12):", KNOWLEDGE_FEATURE_NAMES)

Pruned knowledge feature names (12): ['mean_power', 'std_power', 'min_power', 'power_jump', 'num_transitions', 'rise_time', 'duty_cycle', 'iqr_power', 'autocorr_peak_strength', 'skewness', 'kurtosis', 'dominant_period']


## 8. Appliance-Level Knowledge Profiles (Power / Temporal / Usage)
Computed **only from `train_df`** ground-truth ON/OFF labels, then frozen. These do not
change per window — they encode each appliance's *typical* behavior, learned once, and are
baked into the model as non-trainable weights (Section 10). This mirrors your requested
`Pa, ΔPa, TaON, TaOFF, Fa, Ha, Da` grouped into Power / Temporal / Usage vectors.

In [ ]:
PROFILE_FEATURE_NAMES = [
    # Power (4)
    "typical_power_on", "delta_power_on_off", "standby_power", "power_std_on",
    # Temporal (4)
    "typical_on_duration_sec", "typical_off_duration_sec", "dominant_cycle_period_sec", "duty_cycle",
    # Usage (4)
    "operations_per_day", "peak_hour_of_day", "hour_concentration", "weekday_weekend_ratio",
]
N_PROFILE_FEATURES = len(PROFILE_FEATURE_NAMES)

def run_lengths(mask):
    if len(mask) == 0:
        return []
    mask = mask.astype(int)
    change_points = np.where(np.diff(mask) != 0)[0] + 1
    segments = np.split(np.arange(len(mask)), change_points)
    return [(mask[seg[0]] == 1, len(seg)) for seg in segments]

def circular_mean_hour(hours):
    if len(hours) == 0:
        return 0.0, 0.0
    radians = hours / 24.0 * 2 * np.pi
    sin_mean, cos_mean = np.mean(np.sin(radians)), np.mean(np.cos(radians))
    R = float(np.sqrt(sin_mean ** 2 + cos_mean ** 2))
    mean_angle = np.arctan2(sin_mean, cos_mean)
    mean_hour = (mean_angle / (2 * np.pi)) * 24
    if mean_hour < 0:
        mean_hour += 24
    return float(mean_hour), R

def compute_appliance_profile(df, app, sample_seconds=6):
    power = df[app].values
    on_mask = df[f"{app}_on"].values.astype(bool)
    idx = df.index

    typical_power_on = float(np.mean(power[on_mask])) if on_mask.sum() > 0 else 0.0
    power_std_on = float(np.std(power[on_mask])) if on_mask.sum() > 0 else 0.0
    standby_power = float(np.mean(power[~on_mask])) if (~on_mask).sum() > 0 else 0.0
    delta_power = abs(typical_power_on - standby_power)

    runs = run_lengths(on_mask)
    on_lengths = [length for is_on, length in runs if is_on]
    off_lengths = [length for is_on, length in runs if not is_on]
    typical_on_duration = float(np.mean(on_lengths)) * sample_seconds if on_lengths else 0.0
    typical_off_duration = float(np.mean(off_lengths)) * sample_seconds if off_lengths else 0.0
    dominant_cycle_period = typical_on_duration + typical_off_duration
    duty_cycle = float(np.mean(on_mask))

    n_days = idx.normalize().nunique()
    operations_per_day = len(on_lengths) / max(n_days, 1)

    rising_edges = np.where((on_mask[1:] == True) & (on_mask[:-1] == False))[0] + 1
    if len(on_mask) > 0 and on_mask[0]:
        rising_edges = np.concatenate(([0], rising_edges))
    event_hours = idx[rising_edges].hour.values.astype(float) if len(rising_edges) > 0 else np.array([])
    peak_hour, hour_concentration = circular_mean_hour(event_hours)

    all_days = idx.normalize()
    weekday_days = all_days[all_days.dayofweek < 5].nunique()
    weekend_days = all_days[all_days.dayofweek >= 5].nunique()
    if len(rising_edges) > 0:
        event_is_weekend = idx[rising_edges].dayofweek >= 5
        events_weekday = int((~event_is_weekend).sum())
        events_weekend = int(event_is_weekend.sum())
    else:
        events_weekday, events_weekend = 0, 0
    rate_weekday = events_weekday / max(weekday_days, 1)
    rate_weekend = events_weekend / max(weekend_days, 1)
    weekday_weekend_ratio = rate_weekday / (rate_weekend + 1e-6)

    return [typical_power_on, delta_power, standby_power, power_std_on,
            typical_on_duration, typical_off_duration, dominant_cycle_period, duty_cycle,
            operations_per_day, peak_hour, hour_concentration, weekday_weekend_ratio]

profile_matrix_raw = np.array(
    [compute_appliance_profile(train_df, app, CONFIG["sample_seconds"]) for app in CONFIG["appliances"]],
    dtype=np.float32
)  # shape (5, 12)

print("Raw appliance profiles (computed from TRAIN split only):\n")
profile_df = pd.DataFrame(profile_matrix_raw, index=CONFIG["appliances"], columns=PROFILE_FEATURE_NAMES)
print(profile_df.round(2))

# Min-max normalize each profile feature across the 5 appliances so all dims are
# comparable scale going into the network. Still derived purely from train data.
p_min = profile_matrix_raw.min(axis=0, keepdims=True)
p_max = profile_matrix_raw.max(axis=0, keepdims=True)
profile_matrix_scaled = (profile_matrix_raw - p_min) / (p_max - p_min + 1e-8)

with open(os.path.join(CONFIG["checkpoint_dir"], "appliance_profiles.pkl"), "wb") as f:
    pickle.dump({"raw": profile_matrix_raw, "scaled": profile_matrix_scaled,
                 "feature_names": PROFILE_FEATURE_NAMES, "appliances": CONFIG["appliances"]}, f)

Raw appliance profiles (computed from TRAIN split only):

                 typical_power_on  delta_power_on_off  standby_power  \
dishwasher            1022.210022         1021.299988           0.91   
washing machine        633.630005          633.130005           0.50   
laptop                  25.680000           19.330000           6.35   
fridge                  94.220001           94.120003           0.10   
office lamp 1           13.850000           13.780000           0.07   

                 power_std_on  typical_on_duration_sec  \
dishwasher        1089.729980              1197.000000   
washing machine    814.869995               339.820007   
laptop              10.190000              2754.620117   
fridge              26.639999              1483.589966   
office lamp 1        0.490000             10442.879883   

                 typical_off_duration_sec  dominant_cycle_period_sec  \
dishwasher                   42869.078125               44066.078125   
washing machine 

## 9. Sliding Windows (sequences, window knowledge, state/duty labels)

In [ ]:
def build_windows(df, window_size, stride, appliances, label_rule):
    agg = df["aggregate"].values
    n_samples = len(agg)
    X_seq, X_know, Y_state, Y_duty = [], [], [], []
    for start in range(0, n_samples - window_size + 1, stride):
        end = start + window_size
        window_agg = agg[start:end]
        if np.isnan(window_agg).any():
            continue
        state_labels, duty_targets = [], []
        for app in appliances:
            on_window = df[f"{app}_on"].values[start:end]
            duty_frac = float(np.mean(on_window))
            is_on = int(on_window.any()) if label_rule[app] == "any" else int(duty_frac > 0.5)
            state_labels.append(is_on)
            duty_targets.append(duty_frac)
        X_seq.append(window_agg)
        X_know.append(compute_window_knowledge(window_agg))
        Y_state.append(state_labels)
        Y_duty.append(duty_targets)
    return (np.array(X_seq), np.array(X_know, dtype=np.float32),
            np.array(Y_state), np.array(Y_duty, dtype=np.float32))

X_train_seq, X_train_know, Y_train_state, Y_train_duty = build_windows(
    train_df, CONFIG["window_size"], CONFIG["window_stride"], CONFIG["appliances"], CONFIG["label_rule"])
X_val_seq, X_val_know, Y_val_state, Y_val_duty = build_windows(
    val_df, CONFIG["window_size"], CONFIG["window_stride"], CONFIG["appliances"], CONFIG["label_rule"])
X_test_seq, X_test_know, Y_test_state, Y_test_duty = build_windows(
    test_df, CONFIG["window_size"], CONFIG["window_stride"], CONFIG["appliances"], CONFIG["label_rule"])

print("Train:", X_train_seq.shape, X_train_know.shape)
print("Val:  ", X_val_seq.shape, X_val_know.shape)
print("Test: ", X_test_seq.shape, X_test_know.shape)

Train: (136958, 100) (136958, 12)
Val:   (20854, 100) (20854, 12)
Test:  (38679, 100) (38679, 12)


## 10. Scaling & Class Weights

In [ ]:
seq_scaler = MinMaxScaler()
X_train_seq_s = seq_scaler.fit_transform(X_train_seq.reshape(-1, 1)).reshape(X_train_seq.shape)[..., np.newaxis]
X_val_seq_s = seq_scaler.transform(X_val_seq.reshape(-1, 1)).reshape(X_val_seq.shape)[..., np.newaxis]
X_test_seq_s = seq_scaler.transform(X_test_seq.reshape(-1, 1)).reshape(X_test_seq.shape)[..., np.newaxis]

know_scaler = MinMaxScaler()
X_train_know_s = know_scaler.fit_transform(X_train_know)
X_val_know_s = know_scaler.transform(X_val_know)
X_test_know_s = know_scaler.transform(X_test_know)

with open(os.path.join(CONFIG["checkpoint_dir"], "scalers.pkl"), "wb") as f:
    pickle.dump({"seq_scaler": seq_scaler, "know_scaler": know_scaler,
                 "knowledge_feature_names": KNOWLEDGE_FEATURE_NAMES, "config": CONFIG}, f)

pos_counts = Y_train_state.sum(axis=0).astype(np.float32)
neg_counts = (Y_train_state.shape[0] - pos_counts).astype(np.float32)
pos_weights_arr = np.clip(neg_counts / np.maximum(pos_counts, 1.0), 1.0, 15.0)
print("Positive-class weights:", dict(zip(CONFIG["appliances"], pos_weights_arr.tolist())))

Positive-class weights: {'dishwasher': 15.0, 'washing machine': 12.824366569519043, 'laptop': 1.0, 'fridge': 1.2308406829833984, 'office lamp 1': 4.555654525756836}


## 11. Model — CNN + BiLSTM + Attention + Per-Appliance Compatibility Fusion
Each appliance gets its own compatibility branch comparing `[gated_context ; K_window_encoding ;
its own frozen profile]`, instead of all 5 appliances sharing one Dense(5) head. The appliance
profile matrix is stored as a **non-trainable** Keras weight (`ApplianceProfileLayer`), so it
cannot be updated by gradient descent — it stays exactly what was computed from train data.

In [ ]:
class ApplianceProfileLayer(layers.Layer):
    """Broadcasts a frozen (n_appliances, n_profile_features) matrix to (batch, n_appliances, n_profile_features)."""
    def __init__(self, profile_matrix, **kwargs):
        super().__init__(**kwargs)
        self.profile_matrix_init = np.array(profile_matrix, dtype=np.float32)

    def build(self, input_shape):
        self.profile_matrix = self.add_weight(
            name="appliance_profiles",
            shape=self.profile_matrix_init.shape,
            initializer=tf.keras.initializers.Constant(self.profile_matrix_init),
            trainable=False,
        )
        super().build(input_shape)

    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]
        return tf.tile(tf.expand_dims(self.profile_matrix, 0), [batch_size, 1, 1])

    def get_config(self):
        config = super().get_config()
        config.update({"profile_matrix": self.profile_matrix_init.tolist()})
        return config

def build_cnn_branch(window_size):
    seq_input = layers.Input(shape=(window_size, 1), name="aggregate_sequence")
    x = layers.Conv1D(32, 5, padding="same", activation="relu")(seq_input)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(64, 5, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(64, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    return seq_input, x

def weighted_bce(pos_weights):
    pw = tf.constant(pos_weights, dtype=tf.float32)
    def loss_fn(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        y_true = tf.cast(y_true, tf.float32)
        bce = -(pw * y_true * tf.math.log(y_pred) + (1 - y_true) * tf.math.log(1 - y_pred))
        return tf.reduce_mean(bce)
    return loss_fn

def build_model(window_size, n_knowledge_features, n_profile_features, n_appliances,
                 pos_weights, duty_weight, knowledge_recon_weight, profile_matrix):
    seq_input, cnn_features = build_cnn_branch(window_size)

    bilstm_out = layers.Bidirectional(layers.LSTM(32, return_sequences=True))(cnn_features)
    bilstm_out = layers.Dropout(0.3)(bilstm_out)
    attn_scores = layers.Dense(1, activation="tanh")(bilstm_out)
    attn_weights = layers.Softmax(axis=1)(attn_scores)
    context = layers.Multiply()([bilstm_out, attn_weights])
    context = layers.Lambda(lambda t: tf.reduce_sum(t, axis=1), name="temporal_attention_pool")(context)

    know_input = layers.Input(shape=(n_knowledge_features,), name="knowledge_vector")
    know_encoded = layers.Dense(32, activation="relu")(know_input)
    know_encoded = layers.Dense(16, activation="relu", name="knowledge_encoding")(know_encoded)

    gate_input = layers.Concatenate()([context, know_encoded])
    knowledge_gate = layers.Dense(64, activation="sigmoid", name="knowledge_gate")(gate_input)
    gated_context = layers.Multiply(name="gated_context")([context, knowledge_gate])

    # --- Per-appliance compatibility branch ---
    profiles_broadcast = ApplianceProfileLayer(profile_matrix, name="appliance_profiles")(seq_input)  # (batch, n_app, n_profile)
    gated_context_rep = layers.RepeatVector(n_appliances)(gated_context)   # (batch, n_app, 64)
    know_encoded_rep = layers.RepeatVector(n_appliances)(know_encoded)     # (batch, n_app, 16)

    compat_input = layers.Concatenate(axis=-1, name="compatibility_input")(
        [gated_context_rep, know_encoded_rep, profiles_broadcast])
    compat_hidden = layers.TimeDistributed(layers.Dense(32, activation="relu"), name="compatibility_hidden")(compat_input)
    compat_hidden = layers.TimeDistributed(layers.Dropout(0.2))(compat_hidden)
    per_appliance_prob = layers.TimeDistributed(layers.Dense(1, activation="sigmoid"), name="per_appliance_sigmoid")(compat_hidden)
    state_out = layers.Reshape((n_appliances,), name="state")(per_appliance_prob)

    # --- Duty + knowledge-reconstruction heads (shared trunk, as before) ---
    fused = layers.Concatenate(name="fused_representation")([gated_context, know_encoded])
    shared = layers.Dense(64, activation="relu")(fused)
    shared = layers.Dropout(0.3)(shared)
    shared = layers.Dense(32, activation="relu")(shared)
    duty_out = layers.Dense(n_appliances, activation="sigmoid", name="duty")(shared)
    know_recon_out = layers.Dense(n_knowledge_features, activation="linear", name="knowledge_recon")(shared)

    model = models.Model(inputs=[seq_input, know_input], outputs=[state_out, duty_out, know_recon_out],
                          name="CNN_BiLSTM_Attention_ApplianceProfileFusion")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss={"state": weighted_bce(pos_weights), "duty": "mse", "knowledge_recon": "mse"},
        loss_weights={"state": 1.0, "duty": duty_weight, "knowledge_recon": knowledge_recon_weight},
        metrics={"state": [tf.keras.metrics.BinaryAccuracy(name="acc"),
                            tf.keras.metrics.Precision(name="precision"),
                            tf.keras.metrics.Recall(name="recall")]}
    )
    return model

model = build_model(CONFIG["window_size"], N_KNOWLEDGE_FEATURES, N_PROFILE_FEATURES,
                     len(CONFIG["appliances"]), pos_weights_arr, CONFIG["duty_loss_weight"],
                     CONFIG["knowledge_recon_loss_weight"], profile_matrix_scaled)
model.summary()

Model: "CNN_BiLSTM_Attention_ApplianceProfileFusion"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ aggregate_sequence  │ (None, 100, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 100, 32)   │        192 │ aggregate_sequen… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 100, 32)   │        128 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 100, 64)   │     10,304 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 100, 64)   │        256 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 100, 64)   │     12,352 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 100, 64)   │        256 │ conv1d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 100, 64)   │     24,832 │ batch_normalizat… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 100, 64)   │          0 │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 100, 1)    │         65 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ softmax (Softmax)   │ (None, 100, 1)    │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ knowledge_vector    │ (None, 12)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 100, 64)   │          0 │ dropout[0][0],    │
│                     │                   │            │ softmax[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 32)        │        416 │ knowledge_vector… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ temporal_attention… │ (None, 64)        │          0 │ multiply[0][0]    │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ knowledge_encoding  │ (None, 16)        │        528 │ dense_1[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 80)        │          0 │ temporal_attenti… │
│ (Concatenate)       │                   │            │ knowledge_encodi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ knowledge_gate      │ (None, 64)        │      5,184 │ concatenate[0][0] │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gated_context       │ (None, 64)        │          0 │ temporal_attenti… │
│ (Multiply)          │                   │            │ knowledge_gate[0

 Total params: 65,407 (255.50 KB)

 Trainable params: 65,027 (254.01 KB)

 Non-trainable params: 380 (1.48 KB)

## 12. Resume-or-Build Helper

In [ ]:
checkpoint_path = os.path.join(CONFIG["checkpoint_dir"], "best_model.keras")
history_path = os.path.join(CONFIG["checkpoint_dir"], "history.json")

def resume_or_build_model():
    if os.path.exists(checkpoint_path):
        print("Existing checkpoint found. Loading instead of retraining from scratch...")
        return tf.keras.models.load_model(
            checkpoint_path,
            custom_objects={"loss_fn": weighted_bce(pos_weights_arr), "ApplianceProfileLayer": ApplianceProfileLayer}
        ), True
    print("No checkpoint found. Building a new model.")
    return build_model(CONFIG["window_size"], N_KNOWLEDGE_FEATURES, N_PROFILE_FEATURES,
                        len(CONFIG["appliances"]), pos_weights_arr, CONFIG["duty_loss_weight"],
                        CONFIG["knowledge_recon_loss_weight"], profile_matrix_scaled), False

model, was_resumed = resume_or_build_model()
print("Resumed from checkpoint." if was_resumed else "Starting fresh training run.")

No checkpoint found. Building a new model.
Starting fresh training run.


## 13. Training (checkpointing, early stopping, LR schedule)

In [ ]:
callback_list = [
    callbacks.ModelCheckpoint(checkpoint_path, monitor="val_loss", save_best_only=True, verbose=1),
    callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, verbose=1),
]

history = model.fit(
    [X_train_seq_s, X_train_know_s],
    {"state": Y_train_state, "duty": Y_train_duty, "knowledge_recon": X_train_know_s},
    validation_data=([X_val_seq_s, X_val_know_s],
                      {"state": Y_val_state, "duty": Y_val_duty, "knowledge_recon": X_val_know_s}),
    epochs=CONFIG["epochs"], batch_size=CONFIG["batch_size"], callbacks=callback_list, verbose=1
)

with open(history_path, "w") as f:
    json.dump(history.history, f)
print("Training complete.")

Epoch 1/50
2140/2140 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - duty_loss: 0.0695 - knowledge_recon_loss: 0.0365 - loss: 0.5485 - state_acc: 0.8389 - state_loss: 0.4680 - state_precision: 0.6361 - state_recall: 0.8493
Epoch 1: val_loss improved from None to 1.51104, saving model to /content/drive/MyDrive/SEM7PROJ/checkpoints_phase4_profile_fusion/best_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/SEM7PROJ/checkpoints_phase4_profile_fusion/best_model.keras
2140/2140 ━━━━━━━━━━━━━━━━━━━━ 81s 31ms/step - duty_loss: 0.0520 - knowledge_recon_loss: 0.0213 - loss: 0.4339 - state_acc: 0.8773 - state_loss: 0.3754 - state_precision: 0.6963 - state_recall: 0.9037 - val_duty_loss: 0.1954 - val_knowledge_recon_loss: 0.0280 - val_loss: 1.5110 - val_state_acc: 0.5690 - val_state_loss: 1.3072 - val_state_precision: 0.1350 - val_state_recall: 0.1157 - learning_rate: 0.0010
Epoch 2/50
2139/2140 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - duty_loss: 0.0409 - knowledge_recon_loss: 0.0106 - loss: 

## 14. Predictions on Validation & Test

In [ ]:
val_state_prob, val_duty_pred, _ = model.predict([X_val_seq_s, X_val_know_s], verbose=0)
test_state_prob, test_duty_pred, _ = model.predict([X_test_seq_s, X_test_know_s], verbose=0)
print("Predictions generated for val and test.")

Predictions generated for val and test.


## 15. Per-Appliance Threshold Tuning (validation only, applied to test)
Maximizes F1 per appliance on the validation split, then applies those fixed thresholds
to the test split. Thresholds are never chosen using test labels.

In [ ]:
def tune_thresholds(y_val_true, y_val_prob, appliances):
    thresholds = np.arange(0.05, 0.96, 0.01)
    best_thresholds = []
    for i, app in enumerate(appliances):
        f1s = [f1_score(y_val_true[:, i], (y_val_prob[:, i] > t).astype(int), zero_division=0) for t in thresholds]
        best_t = thresholds[int(np.argmax(f1s))]
        best_thresholds.append(best_t)
        print(f"{app:20s} best threshold: {best_t:.2f}  (val F1 at this threshold: {max(f1s):.3f})")
    return np.array(best_thresholds)

best_thresholds = tune_thresholds(Y_val_state, val_state_prob, CONFIG["appliances"])

with open(os.path.join(CONFIG["checkpoint_dir"], "best_thresholds.pkl"), "wb") as f:
    pickle.dump({"thresholds": best_thresholds, "appliances": CONFIG["appliances"]}, f)

dishwasher           best threshold: 0.92  (val F1 at this threshold: 0.831)
washing machine      best threshold: 0.95  (val F1 at this threshold: 0.940)
laptop               best threshold: 0.26  (val F1 at this threshold: 0.934)
fridge               best threshold: 0.46  (val F1 at this threshold: 0.969)
office lamp 1        best threshold: 0.76  (val F1 at this threshold: 0.789)


## 16. Dwell-Time Correction (applied after thresholding)

In [ ]:
def apply_min_dwell_correction(state_sequence, min_dwell):
    corrected = state_sequence.copy()
    n = len(corrected)
    i = 0
    while i < n:
        j = i
        while j < n and corrected[j] == corrected[i]:
            j += 1
        run_length = j - i
        if run_length < min_dwell and 0 < i and j < n:
            if corrected[i - 1] == corrected[j] and corrected[i] != corrected[j]:
                corrected[i:j] = corrected[j]
        i = j
    return corrected

state_pred_test_fixed = (test_state_prob > 0.5).astype(int)                 # old fixed threshold, for comparison
state_pred_test_tuned = (test_state_prob > best_thresholds).astype(int)     # new tuned thresholds

for pred_arr in (state_pred_test_fixed, state_pred_test_tuned):
    for i, app in enumerate(CONFIG["appliances"]):
        pred_arr[:, i] = apply_min_dwell_correction(pred_arr[:, i], CONFIG["min_dwell_windows"])

print("Dwell correction applied to both fixed-threshold and tuned-threshold predictions.")

Dwell correction applied to both fixed-threshold and tuned-threshold predictions.


## 17. Evaluation — Fixed 0.5 Threshold vs. Tuned Thresholds

In [ ]:
def report_metrics(y_true, y_pred, appliances, label):
    print(f"\n=== {label} ===")
    print(classification_report(y_true, y_pred, target_names=appliances, zero_division=0))
    for i, app in enumerate(appliances):
        p = precision_score(y_true[:, i], y_pred[:, i], zero_division=0)
        r = recall_score(y_true[:, i], y_pred[:, i], zero_division=0)
        f1 = f1_score(y_true[:, i], y_pred[:, i], zero_division=0)
        acc = accuracy_score(y_true[:, i], y_pred[:, i])
        print(f"  {app:20s} | Precision: {p:.3f} | Recall: {r:.3f} | F1: {f1:.3f} | Acc: {acc:.3f}")
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    print(f"\n  MACRO-F1: {macro_f1:.4f}")
    return macro_f1

macro_f1_fixed = report_metrics(Y_test_state, state_pred_test_fixed, CONFIG["appliances"], "Fixed threshold (0.5)")
macro_f1_tuned = report_metrics(Y_test_state, state_pred_test_tuned, CONFIG["appliances"], "Tuned per-appliance thresholds")

print(f"\nMacro-F1 improvement from threshold tuning: {macro_f1_tuned - macro_f1_fixed:+.4f}")


=== Fixed threshold (0.5) ===
                 precision    recall  f1-score   support

     dishwasher       0.53      0.98      0.69       948
washing machine       0.83      0.99      0.90      2798
         laptop       0.93      0.93      0.93     20429
         fridge       0.97      0.98      0.98     17180
  office lamp 1       0.53      0.92      0.67      5901

      micro avg       0.85      0.95      0.90     47256
      macro avg       0.76      0.96      0.83     47256
   weighted avg       0.88      0.95      0.91     47256
    samples avg       0.68      0.73      0.69     47256

  dishwasher           | Precision: 0.531 | Recall: 0.983 | F1: 0.690 | Acc: 0.978
  washing machine      | Precision: 0.832 | Recall: 0.991 | F1: 0.905 | Acc: 0.985
  laptop               | Precision: 0.929 | Recall: 0.926 | F1: 0.927 | Acc: 0.924
  fridge               | Precision: 0.974 | Recall: 0.979 | F1: 0.976 | Acc: 0.979
  office lamp 1        | Precision: 0.528 | Recall: 0.923 | F1: 

## 18. Active-Time Estimation (using tuned thresholds)

In [ ]:
window_minutes = CONFIG["window_size"] * CONFIG["sample_seconds"] / 60.0
print(f"{'Appliance':20s} {'Predicted (min)':>16s} {'Ground truth (min)':>20s} {'Error (min)':>14s} {'% Error':>10s}")
for i, app in enumerate(CONFIG["appliances"]):
    pred_minutes = float(np.sum(test_duty_pred[:, i] * state_pred_test_tuned[:, i]) * window_minutes)
    true_minutes = float(np.sum(Y_test_duty[:, i]) * window_minutes)
    error = pred_minutes - true_minutes
    pct_error = (error / true_minutes * 100) if true_minutes > 0 else float("nan")
    print(f"{app:20s} {pred_minutes:16.1f} {true_minutes:20.1f} {error:14.1f} {pct_error:9.1f}%")

Appliance             Predicted (min)   Ground truth (min)    Error (min)    % Error
dishwasher                     7845.2               9207.5        -1362.3     -14.8%
washing machine               24206.5              25955.5        -1749.0      -6.7%
laptop                       185556.7             203642.9       -18086.2      -8.9%
fridge                       153784.0             171796.5       -18012.5     -10.5%
office lamp 1                 45843.4              58945.6       -13102.2     -22.2%


## 19. Save Final Model

In [ ]:
final_model_path = os.path.join(CONFIG["checkpoint_dir"], "final_model.keras")
model.save(final_model_path)
print("Final model saved to:", final_model_path)
print("Best model (lowest val_loss) is at:", checkpoint_path)
print("Tuned thresholds saved to:", os.path.join(CONFIG["checkpoint_dir"], "best_thresholds.pkl"))
print("Appliance profiles saved to:", os.path.join(CONFIG["checkpoint_dir"], "appliance_profiles.pkl"))

Final model saved to: /content/drive/MyDrive/SEM7PROJ/checkpoints_phase4_profile_fusion/final_model.keras
Best model (lowest val_loss) is at: /content/drive/MyDrive/SEM7PROJ/checkpoints_phase4_profile_fusion/best_model.keras
Tuned thresholds saved to: /content/drive/MyDrive/SEM7PROJ/checkpoints_phase4_profile_fusion/best_thresholds.pkl
Appliance profiles saved to: /content/drive/MyDrive/SEM7PROJ/checkpoints_phase4_profile_fusion/appliance_profiles.pkl
